In [223]:
import pandas as pd
import numpy as np
import os
import re

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 2000)
pd.set_option('display.expand_frame_repr', False)
pd.set_option("display.max_colwidth", None)  # or a large number like 2000


In [224]:
print(os.getcwd())

/run/media/christianm/nvme_storage/Projects/Repos/kaggle_sms_spam_collection/notebooks


In [225]:
df = pd.read_csv("../data/raw/spam.csv", encoding="latin1")

In [226]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 217.8 KB


In [227]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives around here though",NaN,NaN,NaN


In [228]:
df[df["Unnamed: 4"].notna()]

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
281,ham,\Wen u miss someone,the person is definitely special for u..... But if the person is so special,why to miss them,"just Keep-in-touch\"" gdeve.."""
1038,ham,"Edison has rightly said, \A fool can ask more questions than a wise man can answer\"" Now you know why all of us are speechless during ViVa.. GM",GN,GE,"GNT:-)"""
2255,ham,I just lov this line: \Hurt me with the truth,I don't mind,i wil tolerat.bcs ur my someone..... But,"Never comfort me with a lie\"" gud ni8 and sweet dreams"""
3525,ham,\HEY BABE! FAR 2 SPUN-OUT 2 SPK AT DA MO... DEAD 2 DA WRLD. BEEN SLEEPING ON DA SOFA ALL DAY,HAD A COOL NYTHO,TX 4 FONIN HON,"CALL 2MWEN IM BK FRMCLOUD 9! J X\"""""
4668,ham,"When I was born, GOD said, \Oh No! Another IDIOT\"". When you were born",GOD said,"\""OH No! COMPETITION\"". Who knew","one day these two will become FREINDS FOREVER!"""
5048,ham,"Edison has rightly said, \A fool can ask more questions than a wise man can answer\"" Now you know why all of us are speechless during ViVa.. GM",GN,GE,"GNT:-)"""


In [229]:
df["Unnamed: 4"][281]

' just Keep-in-touch\\" gdeve.."'

# Data Cleaning
## Replacing NA with empty string - then concat the strings to one

In [230]:
df.fillna("", inplace=True)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",,,
1,ham,Ok lar... Joking wif u oni...,,,
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,,,
3,ham,U dun say so early hor... U c already then say...,,,
4,ham,"Nah I don't think he goes to usf, he lives around here though",,,
...,...,...,...,...,...
5567,spam,"This is the 2nd time we have tried 2 contact u. U have won the å£750 Pound prize. 2 claim is easy, call 087187272008 NOW1! Only 10p per minute. BT-national-rate.",,,
5568,ham,Will Ì_ b going to esplanade fr home?,,,
5569,ham,"Pity, * was in mood for that. So...any other suggestions?",,,
5570,ham,The guy did some bitching but I acted like i'd be interested in buying something else next week and he gave it to us for free,,,


In [231]:

df["text"] = (
    df[["v2", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]]
      .fillna("")                 # avoid "nan" strings
      .astype(str)
      .agg(" ".join, axis=1)
)

In [232]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4,text
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",,,,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...,,,,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,,,,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,ham,U dun say so early hor... U c already then say...,,,,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though",,,,"Nah I don't think he goes to usf, he lives around here though"


In [233]:
df["target"] = df["v1"]
df.drop(columns=["v1", "v2", "Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], inplace=True)

In [234]:
df.head()

,text,target
0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives around here though",ham


In [235]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5572 non-null   str  
 1   target  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


In [236]:
df["target"].value_counts()

target
ham     4825
spam     747
Name: count, dtype: int64

In [237]:
for i in range(0, 100):
    print(i, df["target"][i], df["text"][i])

0 ham Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...   
1 ham Ok lar... Joking wif u oni...   
2 spam Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's   
3 ham U dun say so early hor... U c already then say...   
4 ham Nah I don't think he goes to usf, he lives around here though   
5 spam FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, å£1.50 to rcv   
6 ham Even my brother is not like to speak with me. They treat me like aids patent.   
7 ham As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune   
8 spam WINNER!! As a valued network customer you have been selected to receivea å£900 prize reward! To claim call 09061701461. C

In [238]:
assert len(df[df["text"].str.strip() == ""]) == 0

## Text Normalization

This then has to be done on the input data of new text as well!

A) normalize unicode / fix mojibake

B) Unescape HTML entities

C) Whitespace cleanup

D) placeholder strategy

In [239]:
df["text"][df["target"] == "spam"]

2             Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's   
5                    FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, å£1.50 to rcv   
8          WINNER!! As a valued network customer you have been selected to receivea å£900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.   
9              Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030   
11                               SIX chances to win CASH! From 100 to 20,000 pounds txt> CSH11 and send to 87575. Cost 150p/day, 6days, 16+ TsandCs apply Reply HL 4 info   
                                                                                        ...                                            

In [252]:
import re
import html
import unicodedata

# ---------- patterns ----------
URL_RE = re.compile(r"(?:https?://\S+|www\.\S+|http://\S+)", flags=re.IGNORECASE)
MONEY_RE = re.compile(r"[£$€]\s*\d{1,3}(?:,\d{3})*(?:\.\d+)?|[£$€]\s*\d+(?:\.\d+)?")
PHONE_RE = re.compile(r"(?<!\w)\d(?:[\s-]?\d){6,}(?!\w)")  # 7+ digits with optional spaces/dashes

# strict number token:
# - 12
# - 1,200
# - 12.5
# - 1,200.50
# avoids weird partials like "1⁄4" fragments
NUM_RE = re.compile(r"\b(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?\b")

# common mojibake traces observed in this dataset
MOJIBAKE_RE = re.compile(r"(?:å£|â£|Ì¼|�|Ã|Â)")


def normalize_text_safe(text: str) -> str:
    if text is None:
        return ""
    text = str(text)

    # 1) targeted replacements FIRST (before NFKC can mutate sequences)
    text = text.replace("å£", "£").replace("â£", "£").replace("Ì¼", "£")

    # 2) html + unicode normalize
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    # 3) non-breaking spaces
    text = text.replace("\u00A0", " ")
    return text


def split_alnum_boundaries(text: str) -> str:
    if text is None:
        return ""
    # POBOXox36504W45WQ -> POBOXox 36504 W 45 WQ
    text = re.sub(r"(?<=[A-Za-z])(?=\d)", " ", text)
    text = re.sub(r"(?<=\d)(?=[A-Za-z])", " ", text)
    return text


def apply_placeholders_v2(text: str) -> str:
    if text is None:
        return ""

    # order matters
    text = URL_RE.sub(" <URL> ", text)
    text = MONEY_RE.sub(" <MONEY> ", text)
    text = PHONE_RE.sub(" <PHONE> ", text)

    # shield unicode fractions before number replacement
    text = re.sub(r"\d+\s*[⁄/]\s*\d+", " <FRACTION> ", text)  # optional token

    # then general numbers
    text = NUM_RE.sub(" <NUM> ", text)
    return text


def clean_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def preprocess_sms_v2(text: str, use_placeholders: bool = True) -> str:
    text = normalize_text_safe(text)
    text = split_alnum_boundaries(text)

    if use_placeholders:
        text = apply_placeholders_v2(text)

    text = clean_whitespace(text)
    return text

In [253]:
text = df["text"].iloc[19]
preprocess_sms(text, True)


0= England v Macedonia - dont miss the goals/team news. Txt ur national team to 87077 eg ENGLAND to 87077 Try:WALES, SCOTLAND 4txt/Ì¼1.20 POBOXox36504W45WQ 16+   
1= England v Macedonia - dont miss the goals/team news. Txt ur national team to 87077 eg ENGLAND to 87077 Try:WALES, SCOTLAND 4txt/£1.20 POBOXox36504W45WQ 16+   
2= England v Macedonia - dont miss the goals/team news. Txt ur national team to 87077 eg ENGLAND to 87077 Try:WALES, SCOTLAND 4 txt/£1.20 POBOXox 36504 W 45 WQ 16+   
3= England v Macedonia - dont miss the goals/team news. Txt ur national team to <NUM> eg ENGLAND to <NUM> Try:WALES, SCOTLAND <NUM> txt/<MONEY> POBOXox <NUM> W <NUM> WQ <NUM>+   
4= England v Macedonia - dont miss the goals/team news. Txt ur national team to <NUM> eg ENGLAND to <NUM> Try:WALES, SCOTLAND <NUM> txt/<MONEY> POBOXox <NUM> W <NUM> WQ <NUM>+


'England v Macedonia - dont miss the goals/team news. Txt ur national team to <NUM> eg ENGLAND to <NUM> Try:WALES, SCOTLAND <NUM> txt/<MONEY> POBOXox <NUM> W <NUM> WQ <NUM>+'

In [254]:
s = df["text"].astype(str)

audit = {
    "rows": len(s),
    "pct_with_mojibake": (s.str.contains(MOJIBAKE_RE).mean() * 100),
    "pct_with_url": (s.str.contains(URL_RE).mean() * 100),
    "pct_with_money_symbol": (s.str.contains(r"[£$€]").mean() * 100),
    "pct_with_phone_like": (s.str.contains(PHONE_RE).mean() * 100),
}

df["text_clean"] = s.map(lambda x: preprocess_sms_v2(x, use_placeholders=False))
df["text_clean_ph"] = s.map(lambda x: preprocess_sms_v2(x, use_placeholders=True))

audit["pct_rows_changed_clean"] = ((df["text_clean"] != s).mean() * 100)
audit["pct_rows_changed_ph"] = ((df["text_clean_ph"] != s).mean() * 100)
audit

{'rows': 5572,
 'pct_with_mojibake': np.float64(4.666188083273511),
 'pct_with_url': np.float64(1.938262742282843),
 'pct_with_money_symbol': np.float64(4.953338119167265),
 'pct_with_phone_like': np.float64(7.483847810480976),
 'pct_rows_changed_clean': np.float64(99.96410624551328),
 'pct_rows_changed_ph': np.float64(99.96410624551328)}

In [255]:
df["text_clean"] = df["text"].map(preprocess_sms)

In [256]:
df.head(20)

,text,target,text_clean,text_clean_ph
0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...","Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,Ok lar... Joking wif u oni...,ham,Ok lar... Joking wif u oni...,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,spam,Free entry in <NUM> a wkly comp to win FA Cup final tkts <NUM> st May <NUM>. Text FA to <NUM> to receive entry question(std txt rate)T&C's apply <PHONE> over <NUM>'s,Free entry in <NUM> a wkly comp to win FA Cup final tkts <NUM> st May <NUM> . Text FA to <NUM> to receive entry question(std txt rate)T&C's apply <PHONE> over <NUM> 's
3,U dun say so early hor... U c already then say...,ham,U dun say so early hor... U c already then say...,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives around here though",ham,"Nah I don't think he goes to usf, he lives around here though","Nah I don't think he goes to usf, he lives around here though"
5,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, å£1.50 to rcv",spam,"FreeMsg Hey there darling it's been <NUM> week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, <MONEY> to rcv","FreeMsg Hey there darling it's been <NUM> week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, <MONEY> to rcv"
6,Even my brother is not like to speak with me. They treat me like aids patent.,ham,Even my brother is not like to speak with me. They treat me like aids patent.,Even my brother is not like to speak with me. They treat me like aids patent.
7,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune,ham,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *<NUM> to copy your friends Callertune,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press * <NUM> to copy your friends Callertune
8,WINNER!! As a valued network customer you have been selected to receivea å£900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.,spam,WINNER!! As a valued network customer you have been selected to receivea <MONEY> prize reward! To claim call <PHONE>. Claim code KL <NUM>. Valid <NUM> hours only.,WINNER!! As a valued network customer you have been selected to receivea <MONEY> prize reward! To claim call <PHONE> . Claim code KL <NUM> . Valid <NUM> hours only.
9,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030,spam,Had your mobile <NUM> months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on <PHONE>,Had your mobile <NUM> months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on <PHONE>


In [257]:
# 1) exact duplicate text rows
dup_mask = df.duplicated(subset=["text"], keep=False)
dups = df.loc[dup_mask, ["text", "target"]].copy()

# 2) conflicting labels for same text
conflicts = (
    dups.groupby("text")["target"]
    .nunique()
    .reset_index(name="n_labels")
    .query("n_labels > 1")
)

print("duplicate rows:", dup_mask.sum())
print("conflicting-text groups:", len(conflicts))

duplicate rows: 684
conflicting-text groups: 0


In [258]:
conflict_texts = set(conflicts["text"])
df_model = df.loc[~df["text"].isin(conflict_texts)].copy()
df_model = df_model.drop_duplicates(subset=["text", "target"]).copy()

In [259]:
df_model.info()

<class 'pandas.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   text           5169 non-null   str  
 1   target         5169 non-null   str  
 2   text_clean     5169 non-null   str  
 3   text_clean_ph  5169 non-null   str  
dtypes: str(4)
memory usage: 201.9 KB


In [260]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df_model["text_clean_ph"]   # or text / text_clean for ablation
y = df_model["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vec = TfidfVectorizer(ngram_range=(1,2), min_df=2)
X_train_vec = vec.fit_transform(X_train)   # fit only on train
X_test_vec = vec.transform(X_test)         # transform test

## Ablation check (raw vs clean vs clean+placeholders)


In [261]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, recall_score

variants = {
    "raw": "text",
    "clean": "text_clean",
    "clean_ph": "text_clean_ph",
}

results = []
for name, col in variants.items():
    X = df_model[col]
    y = df_model["target"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    clf = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2)),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ])

    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)

    spam_recall = recall_score(y_test, pred, pos_label="spam")
    spam_f1 = f1_score(y_test, pred, pos_label="spam")
    weighted_f1 = f1_score(y_test, pred, average="weighted")

    results.append((name, spam_recall, spam_f1, weighted_f1))

results

[('raw', 0.9083969465648855, 0.9296875, 0.9824166961552256),
 ('clean', 0.9389312977099237, 0.924812030075188, 0.9807816038506586),
 ('clean_ph', 0.9389312977099237, 0.924812030075188, 0.9807816038506586)]

# Features

Ideas:
- percentage of uppercase to text length
- Placeholder counts (core signal): num_count, url_count, money_count, phone_count, plus binary flags like has_url, has_money.
- Message length features: character count, token count, average token length, placeholder ratio (placeholder_tokens / total_tokens).
- Spam-trigger token counts: count of words like free, win, claim, urgent, offer, prize, call, txt, stop.
- N-gram TF-IDF: word unigrams+bigrams and char 3–5 grams (char n-grams are very strong for noisy SMS text).
- Digit/alnum pattern intensity: number of mixed alnum tokens (x5, 4u), uppercase token count, punctuation burst count (!, ?, repeated symbols).
- Call-to-action patterns: presence of imperative phrases (call now, click, reply, text back), and short-code patterns.
- Lexical diversity: unique-token ratio and repeated-token ratio (spam often has repetitive wording).
- Position-aware placeholder features: whether url, money, or phone appears near message start/end.
- Rare token features: count of very low-frequency tokens (often promo codes, obfuscations).
- Classical metadata flags: starts with greeting, contains currency symbol, contains deadline language (today, now, last chance).